# Forecast To Inventory Policy

P10/P50/P90 forecasts are converted into proposed min, max, reorder point, and order quantity.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name != "v7_rm_pm_forecast_planning":
    ROOT = Path("Ai miroservices/modeling/v7_rm_pm_forecast_planning").resolve()
OUT = ROOT / "outputs"
pd.set_option("display.max_columns", 80)

In [2]:
forecasts = pd.read_csv(OUT / "forecast_results_v7.csv")
forecasts.head(20)

,material_id,material_code,description,material_type,warehouse_id,warehouse_code,forecast_period,horizon,model_name,forecast_p10,forecast_p50,forecast_p90,method
0,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-02-01,1,V7_RM_PM_DIRECT,36.04,74.91,177.74,lightgbm_global_rm_pm
1,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-03-01,2,V7_RM_PM_DIRECT,44.71,92.93,220.52,lightgbm_global_rm_pm
2,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-04-01,3,V7_RM_PM_DIRECT,50.65,105.26,249.78,lightgbm_global_rm_pm
3,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-05-01,4,V7_RM_PM_DIRECT,37.96,78.89,187.20,lightgbm_global_rm_pm
4,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-06-01,5,V7_RM_PM_DIRECT,48.26,100.29,237.99,lightgbm_global_rm_pm
5,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-07-01,6,V7_RM_PM_DIRECT,40.40,83.97,199.25,lightgbm_global_rm_pm
6,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-08-01,7,V7_RM_PM_DIRECT,31.38,65.22,154.75,lightgbm_global_rm_pm
7,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-09-01,8,V7_RM_PM_DIRECT,31.36,65.17,154.65,lightgbm_global_rm_pm
8,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-10-01,9,V7_RM_PM_DIRECT,24.46,50.83,120.62,lightgbm_global_rm_pm
9,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-11-01,10,V7_RM_PM_DIRECT,23.78,49.43,117.29,lightgbm_global_rm_pm


In [3]:
policy = pd.read_csv(OUT / "inventory_policy_recommendations_v7.csv")
policy.head(30)

,material_id,material_code,description,material_type,horizon_months,forecast_p10_sum,forecast_p50_sum,forecast_p90_sum,avg_monthly_p50,on_hand_qty,available_qty,current_min_stock,current_max_stock,current_reorder_point,inventory_lead_time_days,units_per_pallet,weight_kg,volume_cm3,storage_type,abc_class,fms_class,intermittency_flag,planning_priority,nonzero_rate,cv,lead_time_days,daily_p50,daily_sigma,lead_demand,safety_stock,proposed_min_stock,proposed_reorder_point,proposed_target_stock,proposed_max_stock,suggested_order_qty,current_pallet_positions,target_pallet_positions,pallet_positions_delta,recommendation_status,rationale
0,5571d3cd-666f-4673-a41e-efeb313da005,100036,CAUSTIC SODA,raw_material,12,405710.19,843215.26,2000885.40,70267.938333,1000.0,1000.0,1347.38,77592.04,21037.24,30.0,1.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.292674,30.0,2342.264611,1728.714985,70267.938333,15623.127182,15623.13,85891.07,858838.39,2016508.53,857838.39,1000.00,2016508.53,2015508.53,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
1,ed5d69d5-7370-4a93-b888-2aa711897187,101054,CALCIUM CARBONATE ( GROUND ),raw_material,12,278979.31,579821.79,1375872.85,48318.482500,1000.0,1000.0,40662.75,254847.39,142635.06,80.0,14.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.326836,80.0,1610.616083,1188.719764,128849.286667,17543.184115,17543.18,146392.47,597364.97,1393416.03,596364.97,71.43,99529.72,99458.29,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
2,861bf563-9747-4508-96c0-5a0c976acbcd,101293,FLUFF UNTREATED - GOLDEN ISLES G4881,raw_material,12,253028.61,525886.67,1247888.94,43823.889167,1314.0,1314.0,4013.32,152770.98,101015.49,75.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.335715,75.0,1460.796306,1078.144864,109559.722917,15406.063887,15406.06,124965.78,541292.73,1263295.00,539978.73,65.70,63164.75,63099.05,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
3,44dd0de1-d770-4f74-bccb-e6fc16f574ca,100098,SORBITOL,raw_material,12,240504.26,499856.49,1186121.28,41654.707500,1080.0,1080.0,36986.31,273141.09,205744.40,90.0,17.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.296887,90.0,1388.490250,1024.779161,124964.122500,16041.149419,16041.15,141005.27,515897.64,1202162.43,514817.64,63.53,70715.44,70651.91,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
4,3670333e-585b-4f10-9d15-912dcb65d820,100108,TALCUM POWDER,raw_material,12,114422.19,237811.50,564308.50,19817.625000,1000.0,1000.0,30613.09,133723.39,36779.07,45.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.476880,45.0,660.587500,487.548453,29726.437500,5396.447841,5396.45,35122.89,243207.95,569704.95,242207.95,50.00,28485.25,28435.25,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
5,2b39aadc-6592-406c-914f-482f4cbb7ab5,100323,BC COLOGNE BULK - IMPORTED,raw_material,12,100892.02,209690.78,497580.22,17474.231667,1000.0,1000.0,4902.22,84031.63,54123.55,70.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.229122,70.0,582.474389,429.896874,40773.207222,5934.679241,5934.68,46707.89,215625.46,503514.90,214625.46,50.00,25175.75,25125.75,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
6,584db23d-9317-457f-9e7d-d7592e043210,101580,SODIUM SILICATE,raw_material,12,101836.15,211653.04,502236.51,17637.753333,1200.0,1200.0,16226.41,86296.06,20468.81,30.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.410001,30.0,587.925111,433.919796,17637.753333,3921.516401,3921.52,21559.27,215574.56,506158.03,214374.56,60.00,25307.90,25247.90,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
7,9a11c556-2240-4f35-ae53-f0c8c8a88fb4,100460,GALAXY LES 70,raw_material,12,87862.79,182611.28,433322.64,15217.606667,1000.0,1000.0,182.02,5314.51,2509.90,60.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.378986,60.0,507.253556,374.379953,30435.213333,4784.892161,4784.89,35220.

In [4]:
policy.groupby("recommendation_status").size()

recommendation_status
APPLY_WITH_APPROVAL    263
HIGH_RISK_REVIEW        17
SAFE_TO_APPLY            8
dtype: int64